# Exploration — Groupe HVAC
Analyse des systèmes de chauffage, climatisation et gaines — ResStock 2025

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')

CONSO_COL = 'out.electricity.total.energy_consumption..kwh'
PIC_ETE   = 'out.qoi.electricity.maximum_daily_peak_summer..kw'
PIC_HIVER = 'out.qoi.electricity.maximum_daily_peak_winter..kw'

for col in [CONSO_COL, PIC_ETE, PIC_HIVER]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Shape:', df.shape)

## Section 1 — Types de systèmes et consommation associée

In [ ]:
# Type chauffage × nb bâtiments + conso moy | Type clim | Combustible × conso
heat = df.groupby('in.hvac_heating_type').agg(
    nb=(CONSO_COL, 'count'), conso_moy=(CONSO_COL, 'mean')
).sort_values('nb').tail(10)

cool = df['in.hvac_cooling_type'].value_counts().sort_values()

fuel = df.groupby('in.heating_fuel').agg(
    nb=(CONSO_COL, 'count'), conso_moy=(CONSO_COL, 'mean')
).sort_values('conso_moy')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Heating type — barres colorées par conso moy
norm = (heat['conso_moy'] - heat['conso_moy'].min()) / (heat['conso_moy'].max() - heat['conso_moy'].min())
colors = plt.cm.RdYlGn_r(norm.values)
axes[0].barh(heat.index, heat['nb'], color=colors)
for bar, (_, row) in zip(axes[0].patches, heat.iterrows()):
    axes[0].text(bar.get_width() + 300, bar.get_y() + bar.get_height()/2,
                 f"{row['conso_moy']:,.0f} kWh", va='center', fontsize=7)
axes[0].set_xlabel('Nb batiments')
axes[0].set_title('Type de chauffage\n(couleur = conso moy : vert=faible, rouge=eleve)', fontweight='bold', fontsize=9)

# 2. Cooling type
axes[1].barh(cool.index, cool.values, color='#74add1', alpha=0.85)
for bar, val in zip(axes[1].patches, cool.values):
    axes[1].text(bar.get_width() + 300, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=7)
axes[1].set_xlabel('Nb batiments')
axes[1].set_title('Type de climatisation', fontweight='bold')

# 3. Combustible × conso moy (vert si sous médiane, rouge si au-dessus)
med = fuel['conso_moy'].median()
colors_f = ['#2ecc71' if v < med else '#e74c3c' for v in fuel['conso_moy']]
axes[2].barh(fuel.index, fuel['conso_moy'], color=colors_f, alpha=0.85)
for bar, (_, row) in zip(axes[2].patches, fuel.iterrows()):
    axes[2].text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
                 f"n={row['nb']:,}", va='center', fontsize=7)
axes[2].set_xlabel('Conso moy (kWh/an)')
axes[2].set_title('Conso moy par combustible\n(vert=faible, rouge=eleve)', fontweight='bold', fontsize=9)
axes[2].axvline(med, color='gray', linestyle='--', linewidth=1)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('HVAC — Types de systemes et consommation electrique associee', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'hvac_types_systemes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegarde : hvac_types_systemes.png')

## Section 2 — Consignes de température (Flexibilité FlexiMax)
Les setpoints définissent la plage de confort. Un offset programmé = potentiel de délestage.

In [ ]:
def parse_f(val):
    m = re.search(r'(\d+)', str(val)) if pd.notna(val) else None
    return int(m.group(1)) if m else None

df['heat_sp'] = df['in.heating_setpoint'].apply(parse_f)
df['cool_sp'] = df['in.cooling_setpoint'].apply(parse_f)

hs = df['heat_sp'].value_counts().sort_index()
cs = df['cool_sp'].value_counts().sort_index()

# % bâtiments avec offset programmé (thermostat avec plage de décalage)
pct_heat_offset = (df['in.heating_setpoint_has_offset'] == 'Yes').mean() * 100
pct_cool_offset = (df['in.cooling_setpoint_has_offset'] == 'Yes').mean() * 100

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# 1. Distribution consigne chauffage
axes[0].bar(hs.index.astype(str), hs.values, color='#f46d43', alpha=0.85, edgecolor='white')
axes[0].axvline(str(int(df['heat_sp'].median())), color='#c0392b', linestyle='--', linewidth=2,
                label=f"Mediane = {int(df['heat_sp'].median())}F")
axes[0].set_xlabel('Consigne chauffage (degF)')
axes[0].set_ylabel('Nb batiments')
axes[0].set_title('Distribution consigne chauffage', fontweight='bold')
axes[0].legend(fontsize=9)

# 2. Distribution consigne climatisation
axes[1].bar(cs.index.astype(str), cs.values, color='#74add1', alpha=0.85, edgecolor='white')
axes[1].axvline(str(int(df['cool_sp'].median())), color='#2980b9', linestyle='--', linewidth=2,
                label=f"Mediane = {int(df['cool_sp'].median())}F")
axes[1].set_xlabel('Consigne climatisation (degF)')
axes[1].set_title('Distribution consigne climatisation', fontweight='bold')
axes[1].legend(fontsize=9)

# 3. % avec offset programmé (indicateur flexibilité)
categories = ['Chauffage\navec offset', 'Chauffage\nsans offset',
              'Clim\navec offset', 'Clim\nsans offset']
values = [pct_heat_offset, 100 - pct_heat_offset, pct_cool_offset, 100 - pct_cool_offset]
colors_o = ['#27ae60', '#bdc3c7', '#27ae60', '#bdc3c7']
axes[2].bar(categories, values, color=colors_o, alpha=0.85, edgecolor='white')
axes[2].set_ylabel('% batiments')
axes[2].set_title('Potentiel de flexibilite\n(offset = thermostat programmable)', fontweight='bold', fontsize=9)
for bar, val in zip(axes[2].patches, values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('HVAC — Consignes de temperature et potentiel de flexibilite (FlexiMax)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'hvac_setpoints_flexibilite.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Offset chauffage : {pct_heat_offset:.1f}% | Offset clim : {pct_cool_offset:.1f}%')
print('Sauvegarde : hvac_setpoints_flexibilite.png')

## Section 3 — Gaines (Ducts) : présence, localisation et impact sur la consommation

In [ ]:
# Présence gaines × conso | Localisation gaines | Fuites × conso
duct_conso = df.groupby('in.hvac_has_ducts')[CONSO_COL].agg(['mean', 'count'])
duct_loc   = df['in.duct_location'].value_counts().sort_values()
duct_leak  = df.groupby('in.duct_leakage_and_insulation')[CONSO_COL].mean().sort_values()

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# 1. Avec vs sans gaines — conso moy
labels_d = duct_conso.index.astype(str)
colors_d = ['#e74c3c' if v > duct_conso['mean'].min() else '#2ecc71' for v in duct_conso['mean']]
bars = axes[0].bar(labels_d, duct_conso['mean'], color=colors_d, alpha=0.85, edgecolor='white')
for bar, (idx, row) in zip(axes[0].patches, duct_conso.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f"n={int(row['count']):,}\n{row['mean']:,.0f} kWh", ha='center', fontsize=8)
axes[0].set_ylabel('Conso moy (kWh/an)')
axes[0].set_title('Conso moy avec vs sans gaines', fontweight='bold')

# 2. Localisation des gaines (où sont-elles installées ?)
axes[1].barh(duct_loc.index, duct_loc.values, color='#5b9bd5', alpha=0.85, edgecolor='white')
for bar, val in zip(axes[1].patches, duct_loc.values):
    axes[1].text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=7)
axes[1].set_xlabel('Nb batiments')
axes[1].set_title('Localisation des gaines', fontweight='bold')

# 3. Fuites × conso (top 12 configurations)
top_leak = duct_leak.tail(12)
norm = (top_leak - top_leak.min()) / (top_leak.max() - top_leak.min())
colors_l = plt.cm.RdYlGn_r(norm.values)
labels_l = [str(v)[:30] for v in top_leak.index]
axes[2].barh(labels_l, top_leak.values, color=colors_l)
axes[2].set_xlabel('Conso moy (kWh/an)')
axes[2].set_title('Fuites gaines × conso moy\n(rouge=plus de pertes)', fontweight='bold', fontsize=9)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('HVAC — Gaines : presence, localisation et impact des fuites sur la consommation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'hvac_gaines.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegarde : hvac_gaines.png')